In [6]:
!pip3 install pybullet
!pip3 install -U imageio moviepy==1.0.3
!git clone https://github.com/bulletphysics/bullet3.git

import time
import math
import pybullet as p
import pybullet_data
import numpy as np
import moviepy.editor as mpy
import time
import pandas as pd
import matplotlib.pyplot as plt
import moviepy.editor as mpy
from base64 import b64encode
from IPython.display import HTML
import numpy as np
import uuid, shutil
from typing import List

WIDTH = 360
HEIGHT = 240
DELTA_TIME = 1. / 240

# -------------------------
# 動画保存用関数
# -------------------------
def save_video(frames, path, fps=30):
    clip = mpy.ImageSequenceClip(frames, fps=fps)
#   clip.write_videofile(path, codec="libx264")
    clip.write_videofile(path, fps=30)

def play_mp4(path):
    mp4 = open(path, 'rb').read()
    url = "data:video/mp4;base64," + b64encode(mp4).decode()
    return HTML("""<video width=400 controls><source src="%s" type="video/mp4"></video>""" % url)
# -------------------------
# PyBullet 初期化
# -------------------------
p.connect(p.DIRECT)  # ← 動画取得OK
p.setAdditionalSearchPath(pybullet_data.getDataPath())
p.setGravity(0, 0, -9.8)

# -------------------------
# 環境
# -------------------------
planeId = p.loadURDF("plane.urdf")

tableId = p.loadURDF(
    "table/table.urdf",
    basePosition=[0.5, 0, -0.65],
    useFixedBase=True
)

# -------------------------
# アーム（KUKA iiwa）
# -------------------------
robotId = p.loadURDF(
    "kuka_iiwa/model.urdf",
    basePosition=[0, 0, 0],
    useFixedBase=True
)

num_joints = p.getNumJoints(robotId)
ee_link_index = 6

# -------------------------
# ボール
# -------------------------
ball_radius = 0.04

collision = p.createCollisionShape(p.GEOM_SPHERE, radius=ball_radius)
visual = p.createVisualShape(
    p.GEOM_SPHERE,
    radius=ball_radius,
    rgbaColor=[1, 0, 0, 1]
)

ballId = p.createMultiBody(
    baseMass=0.1,
    baseCollisionShapeIndex=collision,
    baseVisualShapeIndex=visual,
    basePosition=[0.6, 0, 0.65]
)

p.changeDynamics(ballId, -1, lateralFriction=0.8, rollingFriction=0.01)

# -------------------------
# 初期姿勢
# -------------------------
initial_joint_positions = [0, 0.3, 0, -1.2, 0, 1.0, 0.5]
for i in range(num_joints):
    p.resetJointState(robotId, i, initial_joint_positions[i])

# -------------------------
# カメラ設定
# -------------------------
#WIDTH, HEIGHT = 640, 480
WIDTH, HEIGHT = 360, 240
view_matrix = p.computeViewMatrix(
    cameraEyePosition=[1.0, 0, 1.0],
    cameraTargetPosition=[0.5, 0, 0.5],
    cameraUpVector=[0, 0, 1]
)

proj_matrix = p.computeProjectionMatrixFOV(
    fov=60,
    aspect=WIDTH / HEIGHT,
    nearVal=0.1,
    farVal=3.0
)

frames = []

def capture_frame():
    img = p.getCameraImage(
        WIDTH,
        HEIGHT,
        viewMatrix=view_matrix,
        projectionMatrix=proj_matrix
    )
    rgb = np.reshape(img[2], (HEIGHT, WIDTH, 4))[:, :, :3]
    frames.append(rgb)

# -------------------------
# ボールを押す動作
# -------------------------
target_positions = [
    [0.55, 0.0, 0.7],
    [0.65, 0.0, 0.7],
]

for target_pos in target_positions:
    target_ori = p.getQuaternionFromEuler([0, math.pi / 2, 0])

    joint_poses = p.calculateInverseKinematics(
        robotId,
        ee_link_index,
        target_pos,
        target_ori
    )

    for _ in range(200):
        for i in range(num_joints):
            p.setJointMotorControl2(
                robotId,
                i,
                p.POSITION_CONTROL,
                joint_poses[i],
                force=500
            )
        p.stepSimulation()
        capture_frame()

# -------------------------
# 余韻
# -------------------------
for _ in range(300):
    p.stepSimulation()
    capture_frame()

# -------------------------
# 動画保存
# -------------------------
save_video(frames, "kuka_push_ball.mp4", fps=30)
print("🎥 動画保存完了: kuka_push_ball.mp4")
play_mp4("kuka_push_ball.mp4")
#p.disconnect()


fatal: destination path 'bullet3' already exists and is not an empty directory.
Moviepy - Building video kuka_push_ball.mp4.
Moviepy - Writing video kuka_push_ball.mp4



Moviepy - Done !
Moviepy - video ready kuka_push_ball.mp4
🎥 動画保存完了: kuka_push_ball.mp4
